## Indlejringer

I vores tidligere eksempel arbejdede vi med højdimensionelle pose-af-ord-vektorer med længde `vocab_size`, og vi konverterede eksplicit fra lavdimensionelle positionelle repræsentationsvektorer til sparsomme en-til-en repræsentationer. Denne en-til-en repræsentation er ikke hukommelseseffektiv, og desuden behandles hvert ord uafhængigt af hinanden, dvs. en-til-en kodede vektorer udtrykker ikke nogen semantisk lighed mellem ord.

I denne enhed vil vi fortsætte med at udforske **News AG** datasættet. For at begynde, lad os indlæse dataene og få nogle definitioner fra den tidligere notesbog.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Hvad er embedding?

Ideen med **embedding** er at repræsentere ord ved lavdimensionelle tætte vektorer, der på en eller anden måde afspejler et ords semantiske betydning. Vi vil senere diskutere, hvordan man bygger meningsfulde word embeddings, men for nu skal vi bare tænke på embeddings som en måde at reducere dimensionen af en ordvektor på. 

Så vil et embeddinglag tage et ord som input og producere en outputvektor med specificeret `embedding_size`. På en måde ligner det meget et `Linear`-lag, men i stedet for at tage en one-hot kodet vektor, kan det tage et ords nummer som input.

Ved at bruge embeddinglaget som det første lag i vores netværk kan vi skifte fra bag-of-words til en **embedding bag**-model, hvor vi først konverterer hvert ord i vores tekst til den tilsvarende embedding og derefter beregner en aggregationsfunktion over alle disse embeddings, såsom `sum`, `average` eller `max`.  

![Billede der viser en embedding-klassificerer for fem sekvensord.](../../../../../translated_images/da/embedding-classifier-example.b77f021a7ee67eee.webp)

Vores klassificeringsneurale netværk vil starte med et embeddinglag, derefter et aggregeringslag og en lineær klassificerer ovenpå:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Håndtering af variabel sekvensstørrelse

Som følge af denne arkitektur ville minibatches til vores netværk skulle oprettes på en bestemt måde. I den tidligere enhed, når der blev brugt bag-of-words, havde alle BoW-tensorer i en minibatch samme størrelse `vocab_size`, uanset den faktiske længde af vores tekstsekvens. Når vi går over til ord-embedding, ville vi ende med et variabelt antal ord i hver tekstprøve, og når vi kombinerer disse prøver til minibatches, skulle vi anvende noget padding.

Dette kan gøres ved hjælp af den samme teknik med at give en `collate_fn` funktion til datakilden:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Træning af indlejringsklassifikator

Nu hvor vi har defineret en korrekt datalaster, kan vi træne modellen ved at bruge træningsfunktionen, vi har defineret i den tidligere enhed:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Bemærk**: Vi træner kun på 25k poster her (mindre end én fuld epoke) for at spare tid, men du kan fortsætte træningen, skrive en funktion til at træne i flere epoker og eksperimentere med læringsrateparameteren for at opnå højere nøjagtighed. Du bør kunne nå en nøjagtighed på omkring 90%.


### EmbeddingBag-lag og repræsentation af variabel-længde sekvenser

I den tidligere arkitektur skulle vi udfylde alle sekvenser, så de fik samme længde for at kunne placeres i en minibatch. Dette er ikke den mest effektive måde at repræsentere variabel-længde sekvenser på – en anden tilgang ville være at bruge en **offset**-vektor, som indeholder offsets for alle sekvenser gemt i en stor vektor.

![Billede, der viser en offset sekvensrepræsentation](../../../../../translated_images/da/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Bemærk**: På billedet ovenfor viser vi en sekvens af tegn, men i vores eksempel arbejder vi med sekvenser af ord. Dog forbliver det generelle princip for at repræsentere sekvenser med en offset-vektor det samme.

For at arbejde med offset-repræsentation bruger vi [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html)-laget. Det er lignende `Embedding`, men det tager indholdvektor og offset-vektor som input, og det inkluderer også et gennemsnitslag, som kan være `mean`, `sum` eller `max`.

Her er et modificeret netværk, der bruger `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

For at forberede datasættet til træning, skal vi levere en konverteringsfunktion, der vil forberede offset-vektoren:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Bemærk, at i modsætning til alle tidligere eksempler accepterer vores netværk nu to parametre: datavektor og forskydningsvektor, som er af forskellige størrelser. Tilsvarende leverer vores dataloader også 3 værdier i stedet for 2: både tekst- og forskydningsvektorer gives som funktioner. Derfor skal vi justere vores træningsfunktion lidt for at tage højde for dette:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantiske indlejringer: Word2Vec

I vores tidligere eksempel lærte modelindlejringslaget at kortlægge ord til vektorrepræsentation, men denne repræsentation havde ikke meget semantisk mening. Det ville være rart at lære en sådan vektorrepræsentation, hvor lignende ord eller synonymer svarer til vektorer, der er tæt på hinanden med hensyn til en eller anden vektorafstand (fx euklidisk afstand).

For at gøre det skal vi foruddanne vores indlejringsmodel på en stor samling tekst på en specifik måde. En af de første måder at træne semantiske indlejringer på kaldes [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Den er baseret på to hovedarkitekturer, der bruges til at producere en distribueret repræsentation af ord:

 - **Continuous bag-of-words** (CBoW) — i denne arkitektur træner vi modellen til at forudsige et ord ud fra den omgivende kontekst. Givet ngrammet $(W_{-2},W_{-1},W_0,W_1,W_2)$ er modellens mål at forudsige $W_0$ ud fra $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** er det modsatte af CBoW. Modellen bruger et omkringliggende vindue af kontekstord til at forudsige det nuværende ord.

CBoW er hurtigere, mens skip-gram er langsommere, men gør et bedre stykke arbejde med at repræsentere sjældne ord.

![Billede, der viser både CBoW og Skip-Gram algoritmer til at konvertere ord til vektorer.](../../../../../translated_images/da/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

For at eksperimentere med word2vec-indlejring foruddannet på Google News-datasættet kan vi bruge **gensim** biblioteket. Nedenfor finder vi de ord, der er mest lignende 'neural'.

> **Bemærk:** Når du først opretter word-vektorer, kan det tage noget tid at downloade dem!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Vi kan også beregne vektorindlejringer fra ordet, som kan bruges til træning af klassifikationsmodellen (vi viser kun de første 20 komponenter af vektoren for klarhedens skyld):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Det fantastiske ved semantiske indlejringer er, at du kan manipulere vektorrepræsentationen for at ændre semantikken. For eksempel kan vi bede om at finde et ord, hvis vektorrepræsentation er så tæt som muligt på ordene *king* og *woman*, og så langt væk som muligt fra ordet *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Både CBoW og Skip-Grams er "predictive" embeddinger, idet de kun tager lokale kontekster i betragtning. Word2Vec udnytter ikke global kontekst. 

**FastText** bygger videre på Word2Vec ved at lære vektorrepræsentationer for hvert ord og de tegn-n-grammer, der findes i hvert ord. Værdierne af repræsentationerne gennemsnitliggøres derefter til en enkelt vektor ved hvert træningstrin. Selvom dette tilføjer en masse ekstra beregning til for-træningen, gør det det muligt for word embeddings at kode sub-ord information. 

En anden metode, **GloVe**, udnytter idéen om en co-occurrence matrix og bruger neurale metoder til at nedbryde co-occurrence matrixen til mere udtryksfulde og ikke-lineære ordvektorer.

Du kan eksperimentere med eksemplet ved at skifte embedding til FastText og GloVe, da gensim understøtter flere forskellige word embedding modeller.


## Brug af Fortrænede Indlejringer i PyTorch

Vi kan modificere eksemplet ovenfor for at forudfylde matrixen i vores indlejringslag med semantiske indlejringer, såsom Word2Vec. Vi skal tage højde for, at vokabularerne i den fortrænede indlejring og vores tekstkorpus sandsynligvis ikke vil matche, så vi initialiserer vægtene for de manglende ord med tilfældige værdier:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Lad os nu træne vores model. Bemærk, at den tid det tager at træne modellen er betydeligt længere end i det tidligere eksempel, på grund af en større embeddingslagsstørrelse og dermed et meget større antal parametre. Også på grund af dette kan det være nødvendigt at træne vores model på flere eksempler, hvis vi vil undgå overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

I vores tilfælde ser vi ikke en stor stigning i nøjagtigheden, hvilket sandsynligvis skyldes meget forskellige ordforråd. 
For at overvinde problemet med forskellige ordforråd kan vi bruge en af følgende løsninger:
* Gen-træne word2vec-modellen på vores ordforråd
* Indlæse vores datasæt med ordforrådet fra den forudtrænede word2vec-model. Ordforrådet, der bruges til at indlæse datasættet, kan angives under indlæsningen.

Den sidstnævnte tilgang virker nemmere, især fordi PyTorch `torchtext`-rammeværket indeholder indbygget støtte til embeddings. Vi kan for eksempel instantiere GloVe-baseret ordforråd på følgende måde:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Den indlæste ordbog har følgende grundlæggende operationer:
* `vocab.stoi` tillader os at konvertere et ord til dets ordbogsindeks
* `vocab.itos` gør det modsatte - konverterer et tal til et ord
* `vocab.vectors` er arrayet af embeddingsvektorer, så for at få embedding for et ord `s` skal vi bruge `vocab.vectors[vocab.stoi[s]]`

Her er et eksempel på manipulation af embeddings for at demonstrere ligningen **kind-man+woman = queen** (jeg måtte justere koefficienten lidt for at få det til at fungere):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

For at træne klassifikatoren ved hjælp af disse embedding, skal vi først kode vores datasæt ved hjælp af GloVe-ordforrådet:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Som vi har set ovenfor, er alle vektorindlejring gemt i `vocab.vectors` matrix. Det gør det super nemt at indlæse disse vægte i vægtene for indlejringslaget ved hjælp af simpel kopiering:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Lad os nu træne vores model og se, om vi får bedre resultater:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

En af grundene til, at vi ikke ser en væsentlig stigning i nøjagtigheden, er, at nogle ord fra vores datasæt mangler i det for-trænede GloVe-ordforråd, og derfor bliver de i praksis ignoreret. For at overvinde dette kan vi træne vores egne indlejring på vores datasæt. 


## Kontekstuelle Indlejringer

En væsentlig begrænsning ved traditionelle fortrænede indlejringsrepræsentationer som Word2Vec er problemet med ordsammenhængsafklaring. Selvom fortrænede indlejringer kan fange noget af betydningen af ord i kontekst, er hver mulig betydning af et ord kodet ind i den samme indlejring. Dette kan skabe problemer i efterfølgende modeller, da mange ord som ordet 'play' har forskellige betydninger afhængigt af den kontekst, de bruges i.

For eksempel har ordet 'play' i disse to forskellige sætninger en ganske forskellig betydning:
- Jeg gik til et **skuespil** på teatret.
- John vil **lege** med sine venner.

De fortrænede indlejringer ovenfor repræsenterer begge disse betydninger af ordet 'play' i den samme indlejring. For at overvinde denne begrænsning skal vi opbygge indlejringer baseret på **sprogsmodellen**, som er trænet på en stor tekstkorpus, og som *ved*, hvordan ord kan sættes sammen i forskellige kontekster. At diskutere kontekstuelle indlejringer er uden for rammerne af denne vejledning, men vi vil vende tilbage til dem, når vi taler om sprogsmodeller i næste enhed.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
